In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

# ===== Load =====
df = pd.read_csv('/content/drive/MyDrive/Datasets for practice/Life Expectancy Data.csv')
df.columns = df.columns.str.strip()

# Country drop karo — 193 categories, One-Hot ke liye impractical
df = df.drop('Country', axis=1)
df = df.dropna(subset=['Life expectancy'])

# ===== X, y — SPLIT SE PEHLE, koi cleaning nahi (Golden Rule!) =====
X = df.drop('Life expectancy', axis=1)
y = df['Life expectancy']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ===== Feature types define karo =====
numeric_features = X.select_dtypes(include='number').columns.tolist()
categorical_features = ['Status']   # sirf Status categorical hai

# ===== Numeric recipe =====
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),   # ab sirf X_train se seekhega
    ('scaler', StandardScaler())
])

# ===== Categorical recipe =====
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

# ===== Sorting Station (ColumnTransformer) =====
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# ===== Models dictionary =====
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(random_state=42)
}

results = []
trained_pipelines = {}

for name, model in models.items():
    final_pipeline = Pipeline(steps=[
        ('preprocessing', preprocessor),
        ('regressor', model)
    ])
    final_pipeline.fit(X_train, y_train)
    y_pred = final_pipeline.predict(X_test)

    results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R2': r2_score(y_test, y_pred)
    })
    trained_pipelines[name] = final_pipeline

comparison_df = pd.DataFrame(results).round(3)
print(comparison_df)

best_model_name = comparison_df.sort_values('R2', ascending=False).iloc[0]['Model']
best_pipeline = trained_pipelines[best_model_name]
print("Best model:", best_model_name)

joblib.dump(best_pipeline, 'life_expectancy_pipeline.pkl')
from google.colab import files
files.download('life_expectancy_pipeline.pkl')

               Model    MAE   RMSE     R2
0  Linear Regression  2.927  3.951  0.819
1   Ridge Regression  2.927  3.954  0.819
2      Random Forest  1.052  1.675  0.968
Best model: Random Forest


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>